In [45]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [27]:
df = pd.read_csv('my_gurgaon_properties_post_feature_selection_v2.csv')

In [28]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Lower
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Medium
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,Higher
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Medium
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Medium


In [29]:
df['furnishing_type'] = df['furnishing_type'].replace({0.0:"unfurnished",1.0:"semifurnished",2.0:"furnished"})
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Lower
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Medium
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,Higher
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Medium
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Medium


In [30]:
X = df.drop(columns=['price'])
y = df['price']

In [31]:
# Applying the log1p transformation to the target variable -> To make the distribution more normal bcos it is right skewed.
y_transformed = np.log1p(y)

## Ordinal Encoding

In [ ]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [53]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(scores.mean(),scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
print(mean_absolute_error(np.expm1(y_test), y_pred))

0.7362632866007919 0.03247179167821135
0.9463324231715656


In [57]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
    ])

    # K-Fold Cross Validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())

    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    output.append(mae)

    return output

In [58]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'Extra Trees Regressor':ExtraTreesRegressor(),
    'Gradient Boosting':GradientBoostingRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'MLP':MLPRegressor(),
    "XgBoost":XGBRegressor()
}

In [49]:
model_output = []
for name, model in model_dict.items():
    model_output.append(scorer(name,model))

In [52]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
10,XgBoost,0.895622,0.524353
5,Random Forest,0.881749,0.533955
6,Extra Trees Regressor,0.866936,0.556151
7,Gradient Boosting,0.873349,0.573923
4,Decision Tree,0.781140,0.694171
9,MLP,0.803796,0.703692
8,AdaBoost,0.753014,0.844300
1,svr,0.763946,0.850015
2,Ridge,0.736266,0.946291
0,linear_reg,0.736263,0.946332


## One Hot Encoding

In [55]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat1', OrdinalEncoder(), columns_to_encode),
        ('cat2', OneHotEncoder(drop='first'), ['sector','agePossession', 'furnishing_type'])
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [56]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(scores.mean(),scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
print(mean_absolute_error(np.expm1(y_test), y_pred))

0.8545569356442749 0.01611693171756894
0.6493215723753918


In [59]:
model_output = []
for name, model in model_dict.items():
    model_output.append(scorer(name,model))

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
6,Extra Trees Regressor,0.896492,0.471272
10,XgBoost,0.894917,0.485758
5,Random Forest,0.889716,0.510508
9,MLP,0.873353,0.546296
7,Gradient Boosting,0.876119,0.566100
0,linear_reg,0.854557,0.649322
2,Ridge,0.854622,0.652650
4,Decision Tree,0.807242,0.711203
1,svr,0.769535,0.834948
8,AdaBoost,0.755003,0.842941
